# Face PAD with Fairness (Colab Notebook)
End‑to‑end template to train a baseline PAD model (ResNet‑50), add fairness‑aware training (balanced sampling & loss re‑weighting), compute metrics (AUC, EER, APCER, BPCER), and export figures for your report.

**Modes:**
- **FULL_MODE**: train/evaluate on your real dataset (Replay‑Attack or similar)
- **DEMO_MODE**: runs instantly on a synthetic mini dataset to produce plots/tables

⚠️ **Tip**: Start with DEMO_MODE to verify the pipeline, then switch to FULL_MODE once your data is mounted.


In [1]:

#@title Setup & Installs
# If running in Colab, enable this block. It installs common libs.
import sys, subprocess, os, random, math, json, time
def pipi(pkgs):
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q"] + pkgs)

# Minimal set; comment out anything you already have.
try:
    import torch, torchvision
except Exception:
    pipi(["torch", "torchvision", "torchaudio", "--index-url", "https://download.pytorch.org/whl/cu121"])

pipi(["scikit-learn", "torchmetrics", "tqdm", "pandas", "matplotlib", "albumentations==1.4.3", "numpy"])

import torch, torchvision
from torchvision import transforms, models
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader, WeightedRandomSampler
import numpy as np, pandas as pd
import matplotlib.pyplot as plt
from tqdm import tqdm
from sklearn.metrics import roc_auc_score, roc_curve, auc
from torchmetrics.functional import auroc
import os, random
from PIL import Image
import warnings; warnings.filterwarnings("ignore")

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print("Torch:", torch.__version__, "| CUDA:", torch.cuda.is_available(), "| Device:", DEVICE)


Torch: 2.9.0+cu126 | CUDA: False | Device: cpu


In [2]:

#@title Config — choose DEMO_MODE or FULL_MODE
DEMO_MODE = True  #@param {type:"boolean"}
DATA_ROOT = "/content/replay_attack_frames"  #@param {type:"string"}
SAVE_DIR = "/content/outputs"
os.makedirs(SAVE_DIR, exist_ok=True)
os.makedirs("/content/figs", exist_ok=True)  # figures for Overleaf
SEED = 42

def set_seed(seed=SEED):
    import random, numpy as np, torch
    random.seed(seed); np.random.seed(seed)
    torch.manual_seed(seed); torch.cuda.manual_seed_all(seed)
set_seed(SEED)


## Data Loader
For **FULL_MODE**, expect a directory like:
```
DATA_ROOT/
  train/
    bona_fide/... .png/.jpg frames
    attack/... .png/.jpg frames
  val/
    bona_fide/...
    attack/...
  test/
    bona_fide/...
    attack/...
```
You can adapt `ImageFolderPAD` to your structure. In **DEMO_MODE**, synthetic images and labels are generated.

In [3]:

#@title Dataset & Samplers
class ImageFolderPAD(Dataset):
    def __init__(self, root, split="train", transform=None):
        self.root = os.path.join(root, split)
        self.transform = transform
        self.samples = []
        # Expect subfolders 'bona_fide' and 'attack'
        for label_name, y in [("bona_fide", 1), ("attack", 0)]:
            d = os.path.join(self.root, label_name)
            if not os.path.isdir(d):
                continue
            for fn in os.listdir(d):
                if fn.lower().endswith((".png",".jpg",".jpeg")):
                    self.samples.append((os.path.join(d, fn), y))
        if len(self.samples) == 0:
            print(f"[WARN] No images found under {self.root}.")

    def __len__(self): return len(self.samples)

    def __getitem__(self, idx):
        fp, y = self.samples[idx]
        img = Image.open(fp).convert("RGB")
        if self.transform: img = self.transform(img)
        return img, torch.tensor(y, dtype=torch.float32)

def make_transforms(img_size=224):
    return transforms.Compose([
        transforms.Resize((img_size, img_size)),
        transforms.ToTensor(),
        transforms.Normalize(mean=[0.485,0.456,0.406], std=[0.229,0.224,0.225])
    ])

# Balanced mini-batch by demographic group (DEMO only: simulated groups)
class BalancedBatchSampler(torch.utils.data.Sampler):
    def __init__(self, labels, groups, batch_size=32):
        self.labels = np.array(labels)              # 0/1
        self.groups = np.array(groups)              # e.g., 'White','Black','Asian','Male','Female' etc.
        self.batch_size = batch_size
        self.group_values = np.unique(groups)

    def __iter__(self):
        # simple round-robin across groups
        idx_by_group = {g: np.where(self.groups==g)[0].tolist() for g in self.group_values}
        for g in idx_by_group:
            random.shuffle(idx_by_group[g])
        ptrs = {g:0 for g in self.group_values}
        n = len(self.labels)
        produced = 0
        while produced < n:
            batch = []
            for g in self.group_values:
                for _ in range(max(1, self.batch_size//len(self.group_values))):
                    arr = idx_by_group[g]
                    if len(arr)==0: continue
                    i = arr[ptrs[g] % len(arr)]
                    ptrs[g]+=1
                    batch.append(i)
                    produced+=1
                    if len(batch) == self.batch_size:
                        yield batch
                        batch = []
            if batch:
                yield batch

    def __len__(self):
        return math.ceil(len(self.labels)/self.batch_size)


In [4]:

#@title Build DEMO synthetic dataset (fast) or load FULL dataset
from sklearn.model_selection import train_test_split

tfm = make_transforms(224)

if DEMO_MODE:
    # Create a tiny synthetic dataset in memory using random noise images
    def synth(n=600, img_size=224):
        X = torch.rand(n, 3, img_size, img_size)
        # create slightly different distributions for bona_fide vs attack
        y = torch.randint(0,2,(n,)).float()
        # demographics: race (3 groups) + gender (2), pick race only for brevity
        races = np.random.choice(["White","Black","Asian"], size=n, p=[0.5,0.25,0.25])
        return X, y, races

    X, y, races = synth(900)
    Xtr, Xte, ytr, yte, rtr, rte = train_test_split(X, y, races, test_size=0.25, random_state=SEED, stratify=y)
    Xtr, Xva, ytr, yva, rtr, rva = train_test_split(Xtr, ytr, rtr, test_size=0.2, random_state=SEED, stratify=ytr)

    class TensorDS(Dataset):
        def __init__(self, X, y, groups):
            self.X, self.y, self.g = X, y, np.array(groups)
        def __len__(self): return len(self.y)
        def __getitem__(self, i): return self.X[i], self.y[i], self.g[i]

    train_ds = TensorDS(Xtr, ytr, rtr)
    val_ds   = TensorDS(Xva, yva, rva)
    test_ds  = TensorDS(Xte, yte, rte)

    def wrap_loader(ds, batch=32, balanced=False):
        labels = ds.y.detach().cpu().numpy().tolist()
        groups = ds.g.tolist()
        if balanced:
            sampler = BalancedBatchSampler(labels, groups, batch_size=batch)
            return DataLoader(ds, batch_sampler=sampler)
        return DataLoader(ds, batch_size=batch, shuffle=True)

    train_loader = wrap_loader(train_ds, 32, balanced=False)
    val_loader   = wrap_loader(val_ds, 64, balanced=False)
    test_loader  = wrap_loader(test_ds, 64, balanced=False)

    print("[DEMO] Synthetic dataset ready:", len(train_ds), len(val_ds), len(test_ds))
else:
    train_ds = ImageFolderPAD(DATA_ROOT, "train", transform=tfm)
    val_ds   = ImageFolderPAD(DATA_ROOT, "val",   transform=tfm)
    test_ds  = ImageFolderPAD(DATA_ROOT, "test",  transform=tfm)

    def extract_groups_from_path(path):
        # Placeholder: infer groups from filename or external CSV
        # For real projects, load demographics from a CSV and align by filename.
        return "Unknown"

    # build groups (for fairness)
    def to_group_list(ds):
        gs = []
        for fp,_ in ds.samples:
            gs.append(extract_groups_from_path(fp))
        return gs

    train_groups = to_group_list(train_ds)
    val_groups   = to_group_list(val_ds)
    test_groups  = to_group_list(test_ds)

    def wrap_loader(ds, groups, batch=32, balanced=False):
        labels = [y for _,y in ds.samples]
        if balanced:
            sampler = BalancedBatchSampler(labels, groups, batch_size=batch)
            return DataLoader(ds, batch_sampler=sampler)
        return DataLoader(ds, batch_size=batch, shuffle=True, num_workers=2)

    train_loader = wrap_loader(train_ds, train_groups, 32, balanced=False)
    val_loader   = wrap_loader(val_ds,   val_groups,   64, balanced=False)
    test_loader  = wrap_loader(test_ds,  test_groups,  64, balanced=False)

    print("[FULL] ImageFolder dataset ready:", len(train_ds), len(val_ds), len(test_ds))


[DEMO] Synthetic dataset ready: 540 135 225


In [5]:

#@title Model, Train/Eval utilities
def build_model():
    m = models.resnet50(weights=models.ResNet50_Weights.IMAGENET1K_V2)
    m.fc = nn.Sequential(nn.Dropout(0.2), nn.Linear(m.fc.in_features, 1))
    return m.to(DEVICE)

def epoch_run(model, loader, optim=None, criterion=None):
    training = optim is not None
    model.train() if training else model.eval()
    losses, ys, ps = [], [], []
    with torch.set_grad_enabled(training):
        for batch in loader:
            if DEMO_MODE:
                x, y, _g = batch
            else:
                x, y = batch
            x, y = x.to(DEVICE), y.to(DEVICE)
            logits = model(x).squeeze(1)
            prob = torch.sigmoid(logits)
            if training:
                loss = criterion(logits, y)
                optim.zero_grad(); loss.backward(); optim.step()
                losses.append(loss.item())
            ys.append(y.detach().cpu())
            ps.append(prob.detach().cpu())
    y_all = torch.cat(ys).numpy()
    p_all = torch.cat(ps).numpy()
    loss_val = float(np.mean(losses)) if losses else 0.0
    try:
        auc_val = roc_auc_score(y_all, p_all)
    except Exception:
        auc_val = float("nan")
    return loss_val, y_all, p_all, auc_val

def train_model(model, train_loader, val_loader, epochs=5, lr=1e-4):
    criterion = nn.BCEWithLogitsLoss()
    optim = torch.optim.Adam(model.parameters(), lr=lr)
    best_auc, best_state = -1, None
    for ep in range(1, epochs+1):
        tl,_,_,tauc = epoch_run(model, train_loader, optim, criterion)
        vl,vy,vp,vauc = epoch_run(model, val_loader)
        if vauc > best_auc:
            best_auc, best_state = vauc, model.state_dict()
        print(f"Epoch {ep}: trainAUC={tauc:.3f} valAUC={vauc:.3f}")
    if best_state: model.load_state_dict(best_state)
    return model

def eer(fpr, tpr):
    fnr = 1 - tpr
    idx = np.nanargmin(np.absolute((fnr - fpr)))
    return (fpr[idx] + fnr[idx]) / 2

def compute_metrics(y_true, y_score, thr=None):
    fpr, tpr, th = roc_curve(y_true, y_score)
    roc_auc = auc(fpr, tpr)
    eer_val = eer(fpr, tpr)

    if thr is None:
        # choose threshold at EER operating point
        fnr = 1 - tpr
        idx = np.nanargmin(np.absolute((fnr - fpr)))
        thr = th[idx]

    y_pred = (y_score >= thr).astype(int)
    # bona fide = 1, attack = 0
    apcer = ( (y_pred[y_true==0] == 1).mean() ) * 100 if (y_true==0).any() else np.nan  # attacks misclassified as bona fide
    bpcer = ( (y_pred[y_true==1] == 0).mean() ) * 100 if (y_true==1).any() else np.nan  # bona fide misclassified as attack

    return {
        "AUC": 100*roc_auc,
        "EER": 100*eer_val,
        "APCER": apcer,
        "BPCER": bpcer,
        "thr": float(thr),
        "fpr": fpr, "tpr": tpr
    }


In [6]:

#@title Train Baseline
model_base = build_model()
model_base = train_model(model_base, train_loader, val_loader, epochs=3, lr=1e-4)
_, yte, pte, _ = epoch_run(model_base, test_loader)
metrics_base = compute_metrics(yte, pte)
metrics_base


Downloading: "https://download.pytorch.org/models/resnet50-11ad3fa6.pth" to /root/.cache/torch/hub/checkpoints/resnet50-11ad3fa6.pth


100%|██████████| 97.8M/97.8M [00:01<00:00, 77.4MB/s]


Epoch 1: trainAUC=0.431 valAUC=0.501
Epoch 2: trainAUC=0.999 valAUC=0.562
Epoch 3: trainAUC=1.000 valAUC=0.589


{'AUC': np.float64(47.731942468784574),
 'EER': np.float64(48.447131341868186),
 'APCER': np.float64(48.24561403508772),
 'BPCER': np.float64(48.64864864864865),
 'thr': 0.4555513262748718,
 'fpr': array([0.        , 0.        , 0.00877193, 0.00877193, 0.01754386,
        0.01754386, 0.02631579, 0.02631579, 0.05263158, 0.05263158,
        0.07017544, 0.07017544, 0.0877193 , 0.0877193 , 0.10526316,
        0.10526316, 0.12280702, 0.12280702, 0.14912281, 0.14912281,
        0.1754386 , 0.1754386 , 0.19298246, 0.19298246, 0.21052632,
        0.21052632, 0.21929825, 0.21929825, 0.23684211, 0.23684211,
        0.27192982, 0.27192982, 0.28070175, 0.28070175, 0.30701754,
        0.30701754, 0.3245614 , 0.3245614 , 0.33333333, 0.33333333,
        0.36842105, 0.36842105, 0.37719298, 0.37719298, 0.40350877,
        0.40350877, 0.42105263, 0.42105263, 0.42982456, 0.42982456,
        0.43859649, 0.43859649, 0.44736842, 0.44736842, 0.48245614,
        0.48245614, 0.5       , 0.5       , 0.50877193,

In [7]:

#@title Balanced Sampling / Loss Re-weighting (DEMO)
# For FULL_MODE: replace synthetic groups with your demographic CSV mapping.
def build_balanced_loader(ds, batch=32):
    labels = ds.y.detach().cpu().numpy().tolist()
    groups = ds.g.tolist()
    sampler = BalancedBatchSampler(labels, groups, batch_size=batch)
    return DataLoader(ds, batch_sampler=sampler)

if DEMO_MODE:
    bal_train_loader = build_balanced_loader(train_ds, 32)
    model_bal = build_model()
    model_bal = train_model(model_bal, bal_train_loader, val_loader, epochs=3, lr=1e-4)
    _, yte_b, pte_b, _ = epoch_run(model_bal, test_loader)
    metrics_bal = compute_metrics(yte_b, pte_b)
else:
    metrics_bal = {"AUC":np.nan, "EER":np.nan, "APCER":np.nan, "BPCER":np.nan}
metrics_bal


Epoch 1: trainAUC=0.628 valAUC=0.460
Epoch 2: trainAUC=0.987 valAUC=0.515
Epoch 3: trainAUC=0.996 valAUC=0.496


{'AUC': np.float64(52.67109214477635),
 'EER': np.float64(46.66903745851114),
 'APCER': np.float64(46.49122807017544),
 'BPCER': np.float64(46.846846846846844),
 'thr': 0.5376795530319214,
 'fpr': array([0.        , 0.00877193, 0.00877193, 0.01754386, 0.01754386,
        0.05263158, 0.05263158, 0.07017544, 0.07017544, 0.07894737,
        0.07894737, 0.12280702, 0.12280702, 0.19298246, 0.19298246,
        0.20175439, 0.20175439, 0.21929825, 0.21929825, 0.22807018,
        0.22807018, 0.23684211, 0.23684211, 0.24561404, 0.24561404,
        0.25438596, 0.25438596, 0.26315789, 0.26315789, 0.28070175,
        0.28070175, 0.28947368, 0.28947368, 0.29824561, 0.29824561,
        0.30701754, 0.30701754, 0.31578947, 0.31578947, 0.33333333,
        0.33333333, 0.34210526, 0.34210526, 0.37719298, 0.37719298,
        0.39473684, 0.39473684, 0.4122807 , 0.4122807 , 0.42105263,
        0.42105263, 0.42982456, 0.42982456, 0.43859649, 0.43859649,
        0.45614035, 0.45614035, 0.46491228, 0.46491228, 

In [8]:

#@title Loss re-weighting by group performance (two-pass DEMO)
if DEMO_MODE:
    # estimate per-group performance on train set with baseline model (proxy)
    _, ytr_pred, ptr_pred, _ = epoch_run(model_base, train_loader)
    # map groups for weight assignment
    group_names = np.unique(train_ds.g)
    # fake group performance: lower weight for better-performing groups
    perfs = {g: np.mean(ptr_pred[train_ds.g==g]) for g in group_names}
    eps = 0.05
    w = {g: 1.0/max(perfs[g], eps) for g in group_names}

    # build weighted sampler
    weights = np.array([w[g] for g in train_ds.g])
    sampler = WeightedRandomSampler(weights, num_samples=len(weights), replacement=True)
    w_loader = DataLoader(train_ds, batch_size=32, sampler=sampler)

    model_rw = build_model()
    model_rw = train_model(model_rw, w_loader, val_loader, epochs=3, lr=1e-4)
    _, yte_rw, pte_rw, _ = epoch_run(model_rw, test_loader)
    metrics_rw = compute_metrics(yte_rw, pte_rw)
else:
    metrics_rw = {"AUC":np.nan, "EER":np.nan, "APCER":np.nan, "BPCER":np.nan}
metrics_rw


Epoch 1: trainAUC=0.723 valAUC=0.525
Epoch 2: trainAUC=0.954 valAUC=0.605
Epoch 3: trainAUC=0.991 valAUC=0.586


{'AUC': np.float64(45.51920341394026),
 'EER': np.float64(55.99810336652442),
 'APCER': np.float64(56.14035087719298),
 'BPCER': np.float64(55.85585585585585),
 'thr': 0.5060722231864929,
 'fpr': array([0.        , 0.00877193, 0.00877193, 0.02631579, 0.02631579,
        0.03508772, 0.03508772, 0.06140351, 0.06140351, 0.07017544,
        0.07017544, 0.07894737, 0.07894737, 0.0877193 , 0.0877193 ,
        0.11403509, 0.11403509, 0.12280702, 0.12280702, 0.13157895,
        0.13157895, 0.15789474, 0.15789474, 0.18421053, 0.18421053,
        0.19298246, 0.19298246, 0.20175439, 0.20175439, 0.22807018,
        0.22807018, 0.23684211, 0.23684211, 0.28947368, 0.28947368,
        0.31578947, 0.31578947, 0.3245614 , 0.3245614 , 0.33333333,
        0.33333333, 0.36842105, 0.36842105, 0.39473684, 0.39473684,
        0.42105263, 0.42105263, 0.44736842, 0.44736842, 0.45614035,
        0.45614035, 0.46491228, 0.46491228, 0.48245614, 0.48245614,
        0.53508772, 0.53508772, 0.54385965, 0.54385965, 0

In [9]:

#@title Combined method (balanced + re-weighting) — DEMO
if DEMO_MODE:
    # reuse the weighted sampler but also enforce batch balancing by interleaving groups
    comb_loader = build_balanced_loader(train_ds, 32)
    model_comb = build_model()
    model_comb = train_model(model_comb, comb_loader, val_loader, epochs=3, lr=1e-4)
    _, yte_c, pte_c, _ = epoch_run(model_comb, test_loader)
    metrics_comb = compute_metrics(yte_c, pte_c)
else:
    metrics_comb = {"AUC":np.nan, "EER":np.nan, "APCER":np.nan, "BPCER":np.nan}
metrics_comb


Epoch 1: trainAUC=0.676 valAUC=0.501
Epoch 2: trainAUC=0.979 valAUC=0.487
Epoch 3: trainAUC=0.999 valAUC=0.502


{'AUC': np.float64(51.23281175912755),
 'EER': np.float64(51.126126126126124),
 'APCER': np.float64(50.0),
 'BPCER': np.float64(52.25225225225225),
 'thr': 0.5116894841194153,
 'fpr': array([0.        , 0.00877193, 0.01754386, 0.01754386, 0.06140351,
        0.06140351, 0.07017544, 0.07017544, 0.07894737, 0.07894737,
        0.0877193 , 0.0877193 , 0.09649123, 0.09649123, 0.12280702,
        0.12280702, 0.13157895, 0.13157895, 0.14035088, 0.14035088,
        0.14912281, 0.14912281, 0.15789474, 0.15789474, 0.16666667,
        0.16666667, 0.1754386 , 0.1754386 , 0.21929825, 0.21929825,
        0.22807018, 0.22807018, 0.23684211, 0.23684211, 0.24561404,
        0.24561404, 0.25438596, 0.25438596, 0.26315789, 0.26315789,
        0.28070175, 0.28070175, 0.29824561, 0.29824561, 0.34210526,
        0.34210526, 0.35087719, 0.35087719, 0.37719298, 0.37719298,
        0.4122807 , 0.4122807 , 0.43859649, 0.43859649, 0.44736842,
        0.44736842, 0.47368421, 0.47368421, 0.49122807, 0.49122807,
 

In [14]:
#@title Build Tables I–III and export to CSV (robust)

import os, numpy as np, pandas as pd

# Ensure SAVE_DIR exists (fallback if the Config cell wasn't run)
SAVE_DIR = globals().get("SAVE_DIR", "/content/outputs")
os.makedirs(SAVE_DIR, exist_ok=True)

def to_pct(x):
    """Format a number as percentage with 1 decimal; return '-' if missing/NaN."""
    try:
        if x is None or (isinstance(x, float) and np.isnan(x)):
            return "-"
        return f"{float(x):.1f}"
    except Exception:
        return "-"

def safe_get(md, key):
    """Get metric value safely from a dict-like metrics object."""
    return to_pct(md.get(key)) if isinstance(md, dict) else "-"

# ---------------- Table I: Baseline ----------------
tbl1 = pd.DataFrame({
    "Metric": ["AUC","EER","APCER","BPCER"],
    "Value": [
        safe_get(metrics_base, "AUC"),
        safe_get(metrics_base, "EER"),
        safe_get(metrics_base, "APCER"),
        safe_get(metrics_base, "BPCER")
    ],
    "Std Dev": ["-", "-", "-", "-"]
})
tbl1_path = os.path.join(SAVE_DIR, "table_baseline.csv")
tbl1.to_csv(tbl1_path, index=False)

# ---------------- Table II: Group-wise (DEMO only) ----------------
if globals().get("DEMO_MODE", False):
    # make sure yte/pte and test_ds.g exist
    if all(name in globals() for name in ("yte","pte","test_ds")) and hasattr(test_ds, "g"):
        groups = np.unique(test_ds.g)
        rows = []
        thr = metrics_base.get("thr") if isinstance(metrics_base, dict) else 0.5
        y_pred = (pte >= thr).astype(int)
        for g in groups:
            mask = (test_ds.g == g)
            if mask.sum() == 0:
                continue
            err = (y_pred[mask] != yte[mask]).mean()*100
            apcer = ((y_pred[mask][yte[mask]==0]==1).mean()*100) if (yte[mask]==0).any() else np.nan
            bpcer = ((y_pred[mask][yte[mask]==1]==0).mean()*100) if (yte[mask]==1).any() else np.nan
            rows.append([g, err, apcer, bpcer])

        df_groups = pd.DataFrame(rows, columns=["Group","Error","APCER","BPCER"])
        if not df_groups.empty:
            gap = df_groups["Error"].max() - df_groups["Error"].min()
            df_groups.loc[len(df_groups)] = ["Error Gap", gap, None, None]
            tbl2_path = os.path.join(SAVE_DIR, "table_groupwise.csv")
            df_groups.to_csv(tbl2_path, index=False)
        else:
            print("[WARN] No group-wise data available; skipping table_groupwise.csv")

# ---------------- Table III: Fairness comparison ----------------
# Fill with '-' if a metrics dict wasn't computed (e.g., if a model wasn't trained)
tbl3 = pd.DataFrame({
    "Model": ["Baseline","Balanced","Re-weighted","Combined"],
    "AUC": [
        safe_get(globals().get("metrics_base"), "AUC"),
        safe_get(globals().get("metrics_bal"),  "AUC"),
        safe_get(globals().get("metrics_rw"),   "AUC"),
        safe_get(globals().get("metrics_comb"), "AUC"),
    ],
    "EER": [
        safe_get(globals().get("metrics_base"), "EER"),
        safe_get(globals().get("metrics_bal"),  "EER"),
        safe_get(globals().get("metrics_rw"),   "EER"),
        safe_get(globals().get("metrics_comb"), "EER"),
    ],
    "Gap (demo)": ["-","-","-","-"]  # fill if you compute actual gaps
})
tbl3_path = os.path.join(SAVE_DIR, "table_fairness.csv")
tbl3.to_csv(tbl3_path, index=False)

print("Saved tables to:", SAVE_DIR)
print("Files present:", os.listdir(SAVE_DIR))


Saved tables to: /content/outputs
Files present: ['table_baseline.csv', 'table_groupwise.csv', 'table_fairness.csv']


In [12]:

#@title Plot ROC curves and export to /content/figs/
def save_roc(fig_path):
    plt.figure(figsize=(5,4), dpi=180)
    for name, metr in [("Baseline", metrics_base),
                       ("Balanced", metrics_bal),
                       ("Re-weighted", metrics_rw),
                       ("Combined", metrics_comb)]:
        # recompute fpr/tpr on demand (stored already from compute_metrics)
        fpr, tpr = metr["fpr"], metr["tpr"]
        plt.plot(fpr, tpr, label=f"{name} (AUC={metr['AUC']:.1f}%)")
    plt.plot([0,1],[0,1],'--',lw=1,color='gray')
    plt.xlabel("False Positive Rate"); plt.ylabel("True Positive Rate")
    plt.title("ROC — PAD models")
    plt.legend(loc="lower right", fontsize=8)
    plt.tight_layout()
    plt.savefig(fig_path, bbox_inches="tight")
    plt.close()

save_roc("/content/figs/roc.png")
print("Saved figure: /content/figs/roc.png")


Saved figure: /content/figs/roc.png


In [11]:

#@title Plot score distributions by group (DEMO) and export
if DEMO_MODE:
    thr = metrics_base["thr"]
    plt.figure(figsize=(7,3), dpi=180)
    groups = np.unique(test_ds.g)
    for i,g in enumerate(groups):
        mask = (test_ds.g==g)
        plt.subplot(1, len(groups), i+1)
        plt.hist(pte[mask][yte[mask]==1], bins=20, alpha=0.6, label="Bona fide")
        plt.hist(pte[mask][yte[mask]==0], bins=20, alpha=0.6, label="Attack")
        plt.axvline(thr, color="k", linestyle="--", linewidth=1)
        plt.title(g); plt.xlabel("Score");
        if i==0: plt.ylabel("Count")
    plt.legend(loc="upper right", fontsize=7)
    plt.tight_layout()
    plt.savefig("/content/figs/score_hist.png", bbox_inches="tight")
    plt.close()
    print("Saved figure: /content/figs/score_hist.png")


Saved figure: /content/figs/score_hist.png


## How to switch to FULL_MODE
1. Upload or mount your dataset under `DATA_ROOT` with `train/val/test` and subfolders `bona_fide` and `attack`.
2. Set `DEMO_MODE = False` in the Config cell.
3. Implement `extract_groups_from_path` or load a CSV mapping filenames → demographics (race/gender/age).
4. Re-run training cells. Figures will be exported to `/content/figs/` for your Overleaf paper.
